# CCDC land-clearing detection over NSW

Continuous Change Detection and Classification (Zhu & Woodcock 2014) run
through `ee.Algorithms.TemporalSegmentation.Ccdc`, over Sentinel-1 and
Sentinel-2 feature stacks, with cover-stratified break rules.

Unlike the omnibus pipeline, this one works across **forest, woodland and
grassland**. CCDC fits each pixel's own harmonic season + trend model and
flags departures from it, so a noisy grassland pixel gets a wider tolerance
than a stable woodland pixel automatically — which is exactly what a single
global change threshold cannot do.


In [ ]:
# Colab only
# !pip install -q earthengine-api geemap
# !git clone -q https://github.com/JonathanMagson/wind_turbine_detection.git
# %cd wind_turbine_detection


In [ ]:
import ee, geemap

PROJECT = 'your-gee-project'   # <-- change me
ee.Authenticate()
ee.Initialize(project=PROJECT)


In [ ]:
from land_clearing.common import aois, masks, stats
from land_clearing.ccdc import collections as colls, rules, segmentation


## 1. AOI, reporting window, and fitting history

CCDC needs history *before* the window you want to report on, to fit the
seasonal model each break is measured against. Two years is the practical
floor — below about 1.5 the harmonic terms are unreliable.


In [ ]:
AOI_NAME = 'hay'          # grassland-dominated Riverina; try 'moree' for woody
START, END = '2023-01-01', '2024-01-01'
HISTORY_YEARS = 2

aoi = aois.aoi_geometry(name=AOI_NAME)
fit_start = ee.Date(START).advance(-HISTORY_YEARS, 'year').format('YYYY-MM-dd').getInfo()
start_year = segmentation.year_fraction(START)
end_year = segmentation.year_fraction(END)
print('fitting from', fit_start, '| reporting', START, 'to', END)


## 2. Build the two feature stacks

S1 and S2 are segmented **separately**, not interleaved. CCDC drops an
observation entirely if any breakpoint band is masked there, so a combined
collection would throw away every radar acquisition that happened to fall
under a cloud.

Bands are scaled so magnitudes are comparable within each stack: S1 is
dB × 100, S2 is index × 10000.


In [ ]:
s1_coll, rel_orbit = colls.s1_stack(aoi, fit_start, END,
                                    orbit_pass='DESCENDING', multilook_px=3)
s2_coll = colls.s2_stack(aoi, fit_start, END, max_cloud_pct=60)

print('S1:', s1_coll.size().getInfo(), 'images on relative orbit', rel_orbit)
print('S2:', s2_coll.size().getInfo(), 'images')


## 3. Segment

`minObservations` is how many consecutive off-model observations confirm a
break. CCDC's default of 6 was set for Landsat's 16-day revisit; at S1's
6–12 days that would demand two to three months of confirmation, so 4 is
the default here.

This is the expensive step — expect minutes, and use a small AOI first.


In [ ]:
MIN_OBS = 4

s1_ccdc = segmentation.run_ccdc(s1_coll, colls.S1_BANDS, min_observations=MIN_OBS)
s2_ccdc = segmentation.run_ccdc(s2_coll, colls.S2_BANDS, min_observations=MIN_OBS)

s1_breaks = segmentation.extract_breaks(s1_ccdc, colls.S1_BANDS, start_year, end_year, prefix='s1_')
s2_breaks = segmentation.extract_breaks(s2_ccdc, colls.S2_BANDS, start_year, end_year, prefix='s2_')
s1_breaks.bandNames().getInfo()


## 4. Cover-stratified rules

The break rule differs by stratum, because the evidence differs:

| Stratum | Rule | Why |
| --- | --- | --- |
| Forest, woodland | S1 VH magnitude below −1.5 dB | Clearing destroys volume scattering; the drop is large and close to specific |
| Grassland | S2 NDVI drop **and** BSI rise | No volume scattering to lose; radar direction is ambiguous, so optical must carry it |
| Cropland | excluded | Sown-crop cycles produce regular breaks that are not clearing |

The fire screen drops detections whose NBR fell far enough to suggest a
burn rather than clearing. It is a first-order screen, not a substitute for
a burnt-area product.


In [ ]:
strata = masks.cover_strata(include_cropland=False)

clearing = rules.clearing_rules(s1_breaks=s1_breaks, s2_breaks=s2_breaks,
                                strata=strata, screen_fire=True)
clearing = rules.apply_mmu(clearing, min_mmu_ha=0.5, scale=10)

total_ha = stats.cleared_area_ha(clearing, aoi, scale=10)
print(f'detected clearing: {total_ha:.1f} ha')


## 5. Where, and in what cover type

This breakdown is the point of the whole exercise — if grassland comes back
as exactly zero, check that `--sensors` included S2, because grassland is
unassessable from radar alone and the pipeline reports 0 rather than
guessing.


In [ ]:
by_stratum = stats.area_by_stratum(clearing, aoi, scale=10)
for name, ha in sorted(by_stratum.items(), key=lambda kv: -kv[1]):
    print(f'{name:12s} {ha:9.2f} ha')

print()
by_year = stats.area_by_year(clearing, aoi, int(START[:4]), int(END[:4]), scale=10)
for y, ha in sorted(by_year.items()):
    print(f'{y}: {ha:9.2f} ha')


## 6. Map it

The `confidence` band is 2 where both sensors broke within 60 days of each
other (and, in grassland, where the radar also moved in the direction a
conversion would produce), 1 where only one sensor supports it.


In [ ]:
m = geemap.Map()
m.centerObject(aoi, 11)

m.addLayer(strata.selfMask().clip(aoi),
           {'min': 1, 'max': 4, 'palette': ['006400', '9ACD32', 'FFD700', 'D2B48C']},
           'strata (forest/woodland/grassland/cropland)', False)
m.addLayer(clearing.select('t_break').selfMask().clip(aoi),
           {'min': int(START[:4]), 'max': int(END[:4]) + 1,
            'palette': ['fee5d9', 'fb6a4a', 'a50f15']},
           'clearing date')
m.addLayer(clearing.select('confidence').eq(2).selfMask().clip(aoi),
           {'palette': ['0000FF']}, 'both sensors agree', False)
m


## 7. Inspect a single pixel

The fastest way to build trust in a detection, and to calibrate the
magnitude thresholds for your own AOI: click a flagged pixel on the map
above, paste its coordinates in, and look at whether the series really does
step where CCDC says it does.


In [ ]:
PT = ee.Geometry.Point([144.80, -34.55])   # <-- a pixel from the map above

series = s1_coll.select('VH').getRegion(PT, 10).getInfo()
header, rows = series[0], series[1:]
vh = [(r[3], r[4] / colls.S1_SCALE) for r in rows if r[4] is not None]
vh.sort()

import datetime
for t, v in vh[:60]:
    print(datetime.datetime.utcfromtimestamp(t / 1000).date(), f'{v:6.2f} dB')


## 8. Command line and validation

```bash
python -m land_clearing.ccdc.detect --aoi hay \
    --start 2023-01-01 --end 2024-01-01 \
    --project your-gee-project --export drive
```

Then validate against SLATS woody vegetation change from the NSW SEED
portal (see the README). Hansen GFC only covers the woody strata, so use it
to check forest and woodland and ignore what it says about grassland.
